In [1]:
import requests
import os
import pygsheets
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv('riot_api_key')

In [2]:
print(API_KEY)

RGAPI-d0577273-1668-4275-92d7-8c69dd0c77ee


In [3]:
game_name = 'LustGuard'
tag_line = 'BR1'

def get_puuid(game_name=None, tag_line=None, API_KEY=None):
    link = f'https://americas.api.riotgames.com/riot/account/v1/accounts/by-riot-id/{game_name}/{tag_line}?api_key={API_KEY}'
    response = requests.get(link)
    return response.json()['puuid']

In [4]:
get_puuid(game_name=game_name, tag_line=tag_line, API_KEY=API_KEY)

'HBT8DijKYQx_mPwj4v44l-KFYPrqoJnesEvS0aucuXDbFM6EzCwxuO9RbTYDP-jHcISlWLTd_m_klA'

In [5]:
# Get Challengers summonerId

def get_ladder(top=None):
    root = 'https://br1.api.riotgames.com/tft/'
    challenger = 'league/v1/challenger?queue=RANKED_TFT'
    grandmaster = 'league/v1/grandmaster?queue=RANKED_TFT'
    master = 'league/v1/master?queue=RANKED_TFT'

    challenger_response = requests.get(root + challenger + '&api_key=' + API_KEY)
    
    challenger_df = pd.DataFrame(challenger_response.json()['entries']).sort_values('leaguePoints', ascending=False).reset_index(drop=True)
    grandmaster_df = pd.DataFrame()
    master_df = pd.DataFrame()
    
    if top > 50:
        grandmaster_response = requests.get(root + grandmaster + '&api_key=' + API_KEY)
        grandmaster_df = pd.DataFrame(grandmaster_response.json()['entries']).sort_values('leaguePoints', ascending=False).reset_index(drop=True)
    
    if top > 150:
        master_response = requests.get(root + master + '&api_key=' + API_KEY)
        master_df = pd.DataFrame(master_response.json()['entries']).sort_values('leaguePoints', ascending=False).reset_index(drop=True)

    ladder = pd.concat([challenger_df, grandmaster_df, master_df])[:top].reset_index(drop=True)
    ladder = ladder.reset_index(drop=False).drop(columns='rank').rename(columns={'index':'rank'})
    ladder['rank'] += 1

    return ladder

In [6]:
get_ladder(top=250)

,rank,summonerId,leaguePoints,wins,losses,veteran,inactive,freshBlood,hotStreak
0,1,h7ZPEa9uBc7YUhKgYYqSrVFK-Kt-lLQeDBKDP66NVROn-w,846,98,37,False,False,True,True
1,2,dZnoRv82nmMzgyxAGTyrbFzg6ncXgyhx-rK5flUBtmtX,758,112,43,False,False,True,True
2,3,HHzxnU-gmYi4jX3TPyB_T-Ktp_y2n4ByVHUcrzDZTF4dx_0,590,134,66,False,False,True,True
3,4,81TNb5AeP0llAXhrPPctqpn29ySkvIF9j_PJHLEOvMDXAeo,568,89,29,False,False,True,False
4,5,oLf5cNJx5xRUhI0_fZ12VgWuXElsidFB-QwL9KMmD3tSVw,543,67,21,False,False,True,True
5,6,MLeFrfIHO3B8-OFoKfL-LUr02U_pRMvVhtG0CYyLC-AbPE...,436,85,35,False,False,True,True
6,7,woA3HVfaq-VCjpLUuSl3iZDszpWqcqks1JwOgW82M-cxsw,415,101,38,False,False,True,False
7,8,tBOFv3W96MehUyRy3CpZpx8W8Q5RNNzHgGamCl1UXnXYPw,407,75,33,False,False,True,False
8,9,F9ZVEnbOVgl61n923nvdU0SvghMKZ8JR-CKt11DJeQJ6,363,102,55,False,False,True,False
9,10,8ZNCE5oDTwbMwVBmhOZzhOxXJampyL-SWWvA1ZkidSFytQ,318,90,45,False,False,True,False


In [7]:
temp_df = get_ladder(top=100)['summonerId']

In [8]:
puuid_dict = {}
def get_puuid(df):
    root = 'https://br1.api.riotgames.com/tft/league/v1/entries/by-summoner/'
    for summoner in temp_df:
        response = requests.get(root + summoner + '?api_key=' + API_KEY)
        if response.status_code == 200:
            data = response.json()
            summoner_puuid = data[0].get('puuid')
            puuid_dict[summoner] = summoner_puuid
        else:
            print(f'Error to retrieve the data from {summoner}: {response.status_code}')
    return puuid_dict

In [9]:
df = get_puuid(temp_df)
df

{'h7ZPEa9uBc7YUhKgYYqSrVFK-Kt-lLQeDBKDP66NVROn-w': 'pcWnqDDTgTHadYjqVTmLW5PMCTbVwUmNC5EAg1G3BDW5luQvVVLN5W6QVmfwQuA9FCAanz2rqy37OA',
 'dZnoRv82nmMzgyxAGTyrbFzg6ncXgyhx-rK5flUBtmtX': 'Kx3lv8SYashXkZ2ohGuBEqflYQLUtPTUWiMRu_kp4y_kQqEEELfPjaIBpVEVRE64e8PLoITCFapPgA',
 'HHzxnU-gmYi4jX3TPyB_T-Ktp_y2n4ByVHUcrzDZTF4dx_0': 'zBZDYfnwAjB9oRgECZJYT-IdK64eoOnZJCgPAAiVZuF4k5O0m40nutV1Ib6M07e8sMdOdmeaiWrIww',
 '81TNb5AeP0llAXhrPPctqpn29ySkvIF9j_PJHLEOvMDXAeo': 'SpFKHnN3eB68VSMcFBmbWx6SndEiTdl08JCQJpeqxW1LB6FLHqXfsC-NsPShulUZRJrqJGGJkX5wzQ',
 'oLf5cNJx5xRUhI0_fZ12VgWuXElsidFB-QwL9KMmD3tSVw': 'SizMrIAeBru_OJhSk82fngkjtO3JysScmEZm0izvYb_9qVBzjcY-TqvKI45kW75petn2jWMtxbANFA',
 'MLeFrfIHO3B8-OFoKfL-LUr02U_pRMvVhtG0CYyLC-AbPEKYAXTv3W0SCg': 'f5WTZYb2F36D5b1by-DqTw-ac-FzynJOxj9yf30K_8HnHTnzz6Sv7kmsGEAir6ALvYOHZ7coDxtr3w',
 'woA3HVfaq-VCjpLUuSl3iZDszpWqcqks1JwOgW82M-cxsw': '8Nj_JtomOJrTtw3pWzTn0kVXhyV0qXNaki-K4P91sDDf5YWhw9xDFMQMEBUp3lLXamVyUguPPCgr1w',
 'tBOFv3W96MehUyRy3CpZpx8W8Q5RNNzHgGamCl1UXnXYPw': 'PTHaf